# Check Data

Data-quality gatekeeper for `input/Sales_Processed.csv`.

Run this **before** `src/create_folds.py` any time the raw extract changes. If anything here looks wrong (missing dates, duplicate rows, an unexpected gap), stop and fix the source data — don't let it flow into training.

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import config

df = pd.read_csv(config.RAW_DATA_FILE)
print('Shape:', df.shape)
print(df.columns.tolist())
df.head()

## Missing values

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing[missing > 0]

## Date integrity

Convert to datetime, sort chronologically, and confirm the series is daily with no duplicate or skipped dates.

In [ ]:
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df = df.sort_values('Date').reset_index(drop=True)

print('Duplicate dates:', df['Date'].duplicated().sum())
print('Start date:', df['Date'].min())
print('End date:', df['Date'].max())

date_diff = df['Date'].diff().dropna()
print('\nNon-daily gaps:')
print(date_diff[date_diff != pd.Timedelta(days=1)].value_counts())

## Target sanity check

In [ ]:
df['Sales'].describe()

## Quick visual check

One look at the raw series — catches obvious breaks, spikes, or flat stretches that summary stats can miss.

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter(x=df['Date'], y=df['Sales'], mode='lines', name='Historical Sales'))
fig.update_layout(title='Historical Daily Sales', xaxis_title='Date', yaxis_title='Sales',
                   template='plotly_white', height=500)
fig.show()

**If this notebook runs clean:** proceed to `src/create_folds.py` to build features and the chronological train/test split.